# Installation

In [ ]:
# At first, we need to install our libraries[with all their dependencies].
!pip install stable-baselines3[extra]
!pip install pettingzoo
!pip install sb3_contrib
!pip install supersuit
!pip install pymunk
!pip install pygame
!pip install ray[rllib]  # This installs both ray and RLlib
# Let's import them to check their version :)
import stable_baselines3 as SB3
print(SB3.__version__)
import pettingzoo as PET
print(PET.__version__)
import sb3_contrib
print(sb3_contrib.__version__)

---

# Import Libraries

In [ ]:
# We will be creating a parallel environment, meaning that each agent acts simultaneously.
import pettingzoo
from pettingzoo import ParallelEnv
from pettingzoo import AECEnv
from pettingzoo.utils import parallel_to_aec, wrappers, agent_selector
from pettingzoo.utils import ClipOutOfBoundsWrapper
from pettingzoo.utils.conversions import parallel_wrapper_fn
from pettingzoo.sisl._utils import Agent

import gymnasium
from gymnasium.utils import EzPickle ,seeding
from gymnasium.spaces import Discrete, MultiDiscrete, Dict, Box
from gymnasium import spaces

import random
import functools
import numpy as np
import pandas as pd
from copy import copy
from math import pi, sin
from random import gauss, uniform

from tabulate import tabulate
import matplotlib.pyplot as plt

---

# Create our Multi-Agent Environment

This is an environment where multiple agents can take action and observe the results simultaneously. It's a parallel setup that allows for all agents' simultaneous actions and observations.

## Base Environment

In [ ]:
class ReservoirContinues:
    def __init__(self, render_mode=None, num_agents: int = 1):
        """The init method takes in environment arguments.

        Should define the following attributes:
        - possible_agents
        - General Parameters
        - timestep
        - Inflow Parameters
        - Reservoir Parameters

        These attributes should not be changed after initialization.
        """
        # Number of Agents
        self.num_agents = num_agents

        # general parameters
        self.ny = 6                               # Number of years
        self.dt = 1/365                           # Time step size
        self.render_mode = render_mode            # render mode

        # Total number of time steps
        self.time_steps = self.ny / self.dt

        # Inflow parameters
        self.inflow_base = 1                      # Base inflow value
        self.use_inflow_sin = True                # Flag indicating whether to use sinusoidal inflow variation
        self.inflow_sin_amp = 0.2                 # Amplitude of sinusoidal inflow variation
        self.inflow_sin2_amp = 0.1                # Amplitude of second sinusoidal inflow variation
        self.use_inflow_rand_walk = True          # Flag indicating whether to use random walk for inflow variation
        self.inflow_rand_walk_sd = 0.005          # Standard deviation of random walk for inflow variation
        self.use_inflow_jump = True               # Flag indicating whether to use random jumps for inflow variation
        self.inflow_jump_prob = 0.01              # Probability of a random jump occurring in inflow
        self.inflow_jump_amp = 0.2                # Amplitude of random jump in inflow

        # Reservoir parameters
        self.max_release = 3                      # Maximum release from reservoir
        self.min_release = 0                      # Minimum release from reservoir
        self.max_storage = 10                     # Maximum storage capacity of reservoir
        self.min_storage = 0                      # Minimum storage capacity of reservoir
        self.target_base = 5                      # Base target storage level
        self.use_target_sin = True                # Flag indicating whether to use sinusoidal variation for target storage
        self.target_sin_amp = 2                   # Amplitude of sinusoidal variation for target storage

        # observation, reward, and termination
        self.A_done = [False for _ in range(self.num_agents)]
        self.A_obs = [None for _ in range(self.num_agents)]
        self.A_rewards = [0 for _ in range(self.num_agents)]
        self.A_info = [None for _ in range(self.num_agents)]

        # Define the action and observation spaces for all of the agents.
        self.get_spaces()

        # seed generator
        self._seed()

        #self.observation_spaces = dict(zip(self.agents, self.obser_spaces))
        #print("self.observation_spaces -> ",self.observation_spaces)
        #self.action_spaces = dict(zip(self.agents, self.acti_spaces))

    def get_spaces(self):
        """
        Define the action and observation spaces for all of the agents.
        - observation space for n agents -> [[storage1,target1],[storage2,target2], ...,[storageN,targetN]]
        - action space for n agents -> [action1, action2, ..., actionN]
        """
        # this is for storage and target -> array[storage, target]
        # Box(0.0, 10.0, (2,), float32)
        # for example -> [3.017763  1.0108011]
                    # -> [storage=3.017763, target=1.0108011]

        obs_space = spaces.Box(
            low=np.float32(self.min_storage),
            high=np.float32(self.max_storage),
            shape=(2,),
            dtype=np.float32,
        )
        # this is for release -> array[release]
        act_space = spaces.Box(
            low=np.float32(self.min_release),
            high=np.float32(self.max_release),
            shape=(1,),
            dtype=np.float32,
        )
        self.observation_space = [obs_space for i in range(self.num_agents)]
        self.action_space = [act_space for i in range(self.num_agents)]

    def _seed(self, seed=None):
        self.np_random, seed = seeding.np_random(seed)
        return [seed]

    def reset(self, seed=None, options=None):
        """Reset set the environment to a starting point.

        It needs to initialize the following attributes:
        - agents
        - timestep
        - reward
        - terminate
        - observation
          - Storage
          - Target
        - infos
          - Inflow

        And must set up the environment so that render(), step(), and observe() can be called without issues.
        """
        #-------------------------------------------------------
        # Some initialization for each agent and for shared parameters
        temp = uniform(self.min_storage, self.max_storage)
        self.shared_storage = [temp for _ in range(self.num_agents)]
        self.seen_storage = [0. for _ in range(self.num_agents)]
        self.shared_inflow = [0. for _ in range(self.num_agents)]
        self.seen_inflow = [0. for _ in range(self.num_agents)]
        self.shared_inflow_rand_walk = 0.
        self.release = [0. for _ in range(self.num_agents)]
        self.demand = [self.target_base for _ in range(self.num_agents)]
        self.update_demand = [self.target_base for _ in range(self.num_agents)]
        self.t = [0 for _ in range(self.num_agents)] # -> # agent time step
        self.T = 0  # -> Learning phase time step
        #-------------------------------------------------------

        #-------------------------------------------------------
        # Get observation
        observe_list, infos_list = self.observe_list(agent_id=0,flag=True)

        self.A_rewards = [0 for _ in range(self.num_agents)]
        #self.control_rewards = [0 for _ in range(self.num_agents)]
        #self.behavior_rewards = [0 for _ in range(self.num_agents)]
        self.A_done = [False for _ in range(self.num_agents)]
        #-------------------------------------------------------
        self.A_obs = observe_list
        self.A_info = infos_list

        return self.A_obs[0]  #-> observation from first agent

    def update_inflow(self, agent_id):
        """Method for updating the inflow of the reservoir each time step (self.t).

        Depending on parameters, this can be:
        (a) constant inflow,
        (b) combination of seasonally varying sinusoids,
        (c) b plus noise in the form of a Gaussian random walk,
        (d) c plus a random jump process.

        :return: None
        """
        inflow = self.inflow_base
        if self.use_inflow_sin:
            inflow += self.inflow_sin_amp * sin(self.t[agent_id] * 2 * pi) + \
                      self.inflow_sin2_amp * sin(self.t[agent_id] * pi)
        if self.use_inflow_rand_walk:
            self.shared_inflow_rand_walk += gauss(0, self.inflow_rand_walk_sd)
            if self.use_inflow_jump:
                if uniform(0, 1) < self.inflow_jump_prob:
                    if uniform(0, 1) < 0.5:
                        self.shared_inflow_rand_walk += self.inflow_jump_amp
                    else:
                        self.shared_inflow_rand_walk -= self.inflow_jump_amp
            inflow += self.shared_inflow_rand_walk
        self.shared_inflow[agent_id] = max(inflow, 0)

    def Update_demand(self, agent_id):
        """Function for updating demand

        - if use_target_sin option turned on.
        - Otherwise it just stays the same, don't do anything.
        - Assume seasonal target is sin with opposite phase of inflows.

        :return: None
        """
        self.demand[agent_id] = self.update_demand[agent_id]
        if self.use_target_sin:
            self.update_demand[agent_id] = self.target_base - self.target_sin_amp * sin(self.t[agent_id] * 2 * pi)

    def update_storage_release(self, target_release, agent_id):
        """Function for updating storage and release.

        - This update is based on target release, to preserve mass balance
        - param target_release: Target release proposed by RL agent.

        :return: None
        """
        self.seen_storage[agent_id] = self.shared_storage[agent_id]
        self.seen_inflow[agent_id] = self.shared_inflow[agent_id]
        self.shared_storage[agent_id] += (self.shared_inflow[agent_id] - target_release)

        if self.shared_storage[agent_id] < self.min_storage:
            self.release[agent_id] = target_release + self.shared_storage[agent_id]
            self.shared_storage[agent_id] = self.min_storage

        elif self.shared_storage[agent_id] > self.max_storage:
            self.release[agent_id] = target_release + (self.shared_storage[agent_id] - self.max_storage)
            self.shared_storage[agent_id] = self.max_storage
        else:
            self.release[agent_id] = target_release

    def update_reward(self, storage, demand, agent_id):
        """Function to calculate reward for RL agent.

        - Reward diminishes based on abs value deviation from target.
        - param storage: Current storage at end of time step (either continuous or discrete)
        - param target: Current storage target (either continuous or discrete)

        :return: None
        """
        self.A_rewards[agent_id] = 10. - abs(storage - demand)

    def update_time(self,agent_id):
        """Function to update the time step, & terminate simulation after ny years.

        :return: None
        """
        self.A_done[agent_id] = self.t[agent_id] > self.ny - 0.00001
        self.t[agent_id] += self.dt

    def step(self, action, agent_id, is_last):
        """Takes in an action for the current agent (specified by agent_selection).

        - Method for updating the reservoir each time step based on prescribed release target.
        - This involves updating the current inflow and seasonally-varying target,
        - potentially modifying the target release to ensure mass balance, updating the storage,
        - calculating the reward based on deviation from target,
        - updating the time step, and terminating the simulation after a preset number of time steps.
        - release_target: The target release proposed by RL agent. generally stored in a single element list.
        - return:
              1: {'storage': self.storage, 'target': self.target}: dict of observed state that is returned to RL agent;
              2: reward: decreases based on abs deviation of storage from target;
              3: terminate: whether episode ended due to time horizon;
              4: {'inflow':self.inflow, 'release':self.release}: additional state info not returned to agent
        """

        # After performing actions, you would typically update the environment state
        # and gather observations, rewards, termination flags, and information for each agent

        # Update environment state
        self.update_inflow(agent_id=agent_id)
        self.Update_demand(agent_id=agent_id)
        self.update_storage_release(target_release=action,agent_id=agent_id)
        self.update_reward(storage=self.shared_storage[agent_id], demand=self.update_demand[agent_id], agent_id=agent_id)
        self.update_time(agent_id=agent_id)
        ob, inf = self.observe_list(agent_id=agent_id,flag=False)
        self.A_obs[agent_id] = [elem for item in ob for elem in (item if isinstance(item, np.ndarray) else [item])]
        self.A_info[agent_id] = [elem for item in inf for elem in (item if isinstance(item, np.ndarray) else [item])]

        if is_last:
          self.T += 1
          self.printing()
        return self.observe(agent_id)


    def render(self):
        pass

    def close(self):
        pass

    def observe(self, agent_id):
        # remember we have [[storage1,target1],[storage2,target2], ...,[storageN,targetN]]
        return np.array(self.A_obs[agent_id], dtype=np.float32)

    def info(self, agent_id):
        return np.array(self.A_info[agent_id], dtype=np.float32)

    def observe_list(self, agent_id, flag):
        obs_list = []
        inf_list = []
        if flag:  #-> [[storage1,target1],[storage2,target2], ...,[storageN,targetN]]
          for i in range(self.num_agents):
            # next state [next_storage, demand]
            obs_list.append([self.shared_storage[i], self.update_demand[i]])
            # previous state [storage,inflow,demand,release,next_inflow,reward]
            inf_list.append([self.seen_storage[i],
                             self.seen_inflow[i],
                             self.demand[i],
                             self.release[i],
                             self.shared_inflow[i],
                             self.A_rewards[i]])
        if not(flag):  #-> [storage[agent_id],target[agent_id]]
            # next state [next_storage, demand]
            obs_list = [self.shared_storage[agent_id], self.update_demand[agent_id]]
            # previous state [storage,inflow,demand,release,next_inflow,reward]
            inf_list = [self.seen_storage[agent_id],
                        self.seen_inflow[agent_id],
                        self.demand[agent_id],
                        self.release[agent_id],
                        self.shared_inflow[agent_id],
                        self.A_rewards[agent_id]]
        return obs_list, inf_list

    def printing(self):
        """Prints observations and actions in a tabular format with two levels of headers."""
        # Define the two-tier headers
        headers = pd.MultiIndex.from_tuples([
            ("Agent Information", "Time Step"),
            ("Agent Information", "Agent ID"),
            ("Agent Information", "Agent Time Step"),
            ("Current State and Decision Information", "Demand"),
            ("Current State and Decision Information", "Observed Storage"),
            ("Current State and Decision Information", "Observed Inflow"),
            ("Current State and Decision Information", "Release"),
            ("Feedback and Next State Information", "Reward"),
            ("Feedback and Next State Information", "Next Storage"),
            ("Feedback and Next State Information", "Updated Demand")
        ])

        data = []
        for i in range(self.num_agents):
            storage = f"{self.seen_storage[i][0]:.2f}" if isinstance(self.seen_storage[i], np.ndarray) else f"{self.seen_storage[i]:.2f}"
            inflow = f"{self.seen_inflow[i]:.2f}" if isinstance(self.seen_inflow[i], np.ndarray) else f"{self.seen_inflow[i]:.2f}"
            release = f"{self.release[i][0]:.2f}" if isinstance(self.release[i], np.ndarray) else f"{self.release[i]:.2f}"
            demand = f"{self.demand[i]:.2f}"  # Assuming demand is already a scalar
            reward = f"{self.A_rewards[i][0]:.2f}" if isinstance(self.A_rewards[i], np.ndarray) else f"{self.A_rewards[i]:.2f}"
            next_storage = f"{self.shared_storage[i][0]:.2f}" if isinstance(self.shared_storage[i], np.ndarray) else f"{self.shared_storage[i]:.2f}"
            next_demand = f"{self.update_demand[i]:.2f}"  # Assuming updated demand is already a scalar

            row = [
                self.T,  # Learning Phase Time Step
                f"Agent-{i+1}",  # Agent ID
                self.t[i],  # Agent Time Step
                demand,  # Demand
                storage,  # Observed Storage
                inflow,  # Observed Inflow
                release,  # Release
                reward,  # Reward
                next_storage,  # Next Observation(Storage)
                next_demand  # Next Observation(Updated Demand)
            ]
            data.append(row)



        # Create DataFrame
        df = pd.DataFrame(data, columns=headers)

        # Styling enhancements
        styles = [
            {'selector': 'td:hover', 'props': [('background-color', '#F08080')]},
            {'selector': '.index_name', 'props': 'font-style: italic; color: darkgrey; font-weight:normal;'},
            {'selector': 'th:not(.index_name)', 'props': 'background-color: cornflowerblue; color: white;'},
            {'selector': 'th', 'props': [('text-align', 'center')]},
            {'selector': 'td', 'props': [('text-align', 'center')]},
            {'selector': '.col2', 'props': [('border-right', '1.5px solid white')]},
            {'selector': '.col6', 'props': [('border-right', '1.5px solid white')]}
        ]

        # Apply the updated styles
        styled_df = df.style.set_table_styles(styles).set_properties(**{'border': '1.5px solid black'}).hide(axis="index")

        # Display the styled DataFrame
        display(styled_df)
        print("----------------------------------------------------------------------------------------------------------------------------------")


## Main Environment

In [ ]:
def ReservoirContinuesMulti(**kwargs):
    env = raw_env(**kwargs)
    env = wrappers.ClipOutOfBoundsWrapper(env)
    env = wrappers.OrderEnforcingWrapper(env)
    return env

parallel_env = parallel_wrapper_fn(ReservoirContinuesMulti)

class raw_env(AECEnv, EzPickle):
    metadata = {
        "render_modes": ["human", "rgb_array"],
        "name": "Reservoir_Continues_V0",
        "is_parallelizable": True,
    }
    def __init__(self, *args, **kwargs):
        EzPickle.__init__(self, *args, **kwargs)
        AECEnv.__init__(self)
        self.env = ReservoirContinues(*args, **kwargs)

        self.agents = ["Agent-" + str(i) for i in range(1, self.env.num_agents + 1)]
        self.possible_agents = self.agents[:]  # -> ['Agent_1', 'Agent_2',..., So on]
        # optional: a mapping between agent name and ID -> {'Agent_1': 0, 'Agent_2': 1, so on...}
        self.agent_name_mapping = dict(zip(self.agents, list(range(self.num_agents))))
        self._agent_selector = agent_selector(self.agents) #-> to manage turn-taking among agents.

        # spaces
        self.action_spaces = dict(zip(self.agents, self.env.action_space))
        self.observation_spaces = dict(zip(self.agents, self.env.observation_space))
        self.has_reset = False

        self.render_mode = self.env.render_mode

    # lru_cache allows observation and action spaces to be memoized, reducing clock cycles required to get each agent's space.
    # If your spaces change over time, remove this line (disable caching).
    #@functools.lru_cache(maxsize=None)
    def observation_space(self, agent):
        # agent argument is -> Agent_1
        return self.observation_spaces[agent]

    # If your spaces change over time, remove this line (disable caching).
    #@functools.lru_cache(maxsize=None)
    def action_space(self, agent):
        # agent argument is -> Agent_1
        return self.action_spaces[agent]

    def convert_to_dict(self, list_of_list):
        return dict(zip(self.agents, list_of_list))

    def reset(self, seed=None, options=None):
        if seed is not None:
            self.env._seed(seed=seed)
        self.has_reset = True
        self.env.reset()
        self.agents = self.possible_agents[:]
        self._agent_selector.reinit(self.agents)
        self.agent_selection = self._agent_selector.next()
        self.rewards = dict(zip(self.agents, [(0) for _ in self.agents]))
        self._cumulative_rewards = dict(zip(self.agents, [(0) for _ in self.agents]))
        self.terminations = dict(zip(self.agents, [False for _ in self.agents]))
        self.truncations = dict(zip(self.agents, [False for _ in self.agents]))
        self.infos = dict(zip(self.agents, [{} for _ in self.agents]))

    def close(self):
        if self.has_reset:
            self.env.close()

    def render(self):
        return self.env.render()

    def step(self, action):
        if (
            self.terminations[self.agent_selection]
            or self.truncations[self.agent_selection]
        ):
            self._was_dead_step(action)
            return

        agent = self.agent_selection
        is_last = self._agent_selector.is_last()
        self.env.step(action, self.agent_name_mapping[agent], is_last)
        A_rewards = [arr.item() if isinstance(arr, np.ndarray) else arr for arr in self.env.A_rewards]
        self.rewards = dict(zip(self.agents, A_rewards))
        self.terminations = dict(zip(self.agents, self.env.A_done))
        self._cumulative_rewards[self.agent_selection] = 0
        self.agent_selection = self._agent_selector.next()
        self._accumulate_rewards()

    def observe(self, agent):
        return self.env.observe(self.agent_name_mapping[agent])

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


---

# Test the Environment

In [ ]:
from pettingzoo.test import parallel_api_test
from pettingzoo.utils import aec_to_parallel
from pettingzoo.utils import ClipOutOfBoundsWrapper

if __name__ == "__main__":
    #env = ReservoirContinues(render_mode=None, num_agents=4)
    env = ReservoirContinuesMulti(render_mode=None, num_agents=2)
    env = ClipOutOfBoundsWrapper(env)
    env = aec_to_parallel(env)
    parallel_api_test(env, num_cycles=10)

    """
    for i in range(1):
      # this is where you would insert your policy
      actions = {agent: env.action_space(agent).sample() for agent in env.agents}
      observations, rewards, terminations, truncations, infos = env.step(actions)
    """

----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


...

----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------
Passed Parallel API test


# Run RL in our Environment

In [ ]:
from __future__ import annotations  # Importing annotations from future for forward references

import glob  # Module to find all the pathnames matching a specified pattern
import os  # Module providing functions for interacting with the operating system
import time  # Module to work with time-related functions

import supersuit as ss  # Supersuit library for modifying gym environments
from stable_baselines3 import PPO  # Importing Proximal Policy Optimization algorithm from stable_baselines3
from stable_baselines3.ppo import MlpPolicy, MultiInputPolicy  # Importing Multi-Layer Perceptron policy for PPO
from sb3_contrib import MaskablePPO
from pettingzoo.utils import aec_to_parallel
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder


def train(
    env_, steps: int = 10_000, seed: int | None = 0, num_agents: int = 1, **env_kwargs):
    # Train function to train an RL agent
    env = env_(render_mode=None,num_agents=num_agents)  # Assigning environment function to env
    #----------------------
    # Included PettingZoo wrappers currently do not support parallel environments,
    # to use them you must convert your environment to AEC, apply the wrapper, and convert back to parallel.
    env = ClipOutOfBoundsWrapper(env)
    env = aec_to_parallel(env)
    #----------------------
    env.reset(seed=seed)

    print(f"Starting training on {str(env.metadata['name'])}.")  # Printing metadata of the environment

    env = ss.pettingzoo_env_to_vec_env_v1(env)  # Converting PettingZoo environment to vectorized environment
    # to run multiple versions of itself in parallel.
    # Playing through the environment multiple times at once makes learning faster and is important to PPOs learning performance.
    env = ss.concat_vec_envs_v1(env, 2, num_cpus=1, base_class="stable_baselines3")

    # our PPO model
    model = PPO(
        MlpPolicy,
        env,
        verbose=1,
    )

    model.learn(total_timesteps=steps)  # Training the model
    policy = model.policy

    # Saving the trained model with a timestamp
    !ls trained_models
    !mkdir trained_models
    model_name = "PPO"  # Specify the name of the model here (e.g., PPO, DQN, etc.)
    env_name = env.unwrapped.metadata.get('name')  # Get the name of the environment
    # Saving the trained model with a timestamp and model/environment name
    timestamp = time.strftime('%Y%m%d-%H%M%S')
    model_save_name = f"{model_name}_{env_name}_{timestamp}"
    policy.save(f"trained_models/{model_save_name}")

    print("Model has been saved.")  # Printing confirmation of model saving

    print(f"Finished training on {str(env.unwrapped.metadata['name'])}.")  # Printing metadata of the finished training

    env.close()  # Closing the environment
    return model_save_name


def eval(env_, name, num_games: int = 100, render_mode: str | None = None, **env_kwargs):
    # Evaluation function to evaluate a trained RL agent
    env = env_(render_mode=render_mode, num_agents=num_agents)  # Creating environment for evaluation

    print(
        f"\nStarting evaluation on {str(env.metadata['name'])} (num_games={num_games}, render_mode={render_mode})"
    )  # Printing metadata of the evaluation environment

    try:
        ## get trained policy from disk
        loaded_policy = MultiInputPolicy.load(f"trained_models/{name}")
    except ValueError:
        print("Policy not found.")  # Printing if policy file not found
        exit(0)  # Exiting if policy file not found

    rewards = {agent: 0 for agent in env.possible_agents}  # Initializing dictionary to store rewards

    # Evaluating the model by playing multiple games
    for i in range(num_games):
        env.reset(seed=i)  # Resetting the environment with different seeds

        # Looping through each agent's turn
        for agent in env.agent_iter():
            obs, reward, termination, truncation, info = env.last()  # Getting observation, reward, termination, truncation, and info

            for a in env.agents:
                rewards[a] += env.rewards[a]  # Accumulating rewards for each agent
            if termination or truncation:  # If episode is terminated or truncated
                break  # Exiting the loop
            else:
                act = loaded_policy.predict(obs, deterministic=True)[0]  # Getting the action from the model

            env.step(act)  # Taking a step in the environment based on the action
    env.close()  # Closing the environment after evaluation

    avg_reward = sum(rewards.values()) / len(rewards.values())  # Calculating average reward
    print("Rewards: ", rewards)  # Printing rewards
    print(f"Avg reward: {avg_reward}")  # Printing average reward
    return avg_reward  # Returning average reward


if __name__ == "__main__":
    # Define the number of agents
    num_agents = 2
    env_ = ReservoirContinuesMulti # Initializing environment

    env_kwargs = {}  # Initializing environment keyword arguments

    # Train a model
    model_name = train(env_, steps=10, seed=0, num_agents=num_agents, **env_kwargs)

    # Evaluate 10 games
    eval(env_, name=model_name, num_games=2, render_mode=None, **env_kwargs)

Starting training on Reservoir_Continues_V0.
Using cpu device


----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


...

----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------------------------------------


KeyboardInterrupt: 